# Pipeline Technical Documentation & Recreation Guide

This document provides a function-level, step-by-step description of each file in the Heatmap-Driven Per-Macroblock QP Offset Encoding Pipeline so that you can easily understand, modify, or recreate it.

---

## 1. Python QP Offset Generator: `qp_map_generator.py`

This script processes input videos frame-by-frame, runs them through a visual saliency model placeholder, min/max normalizes the maps to the `[0.0, 1.0]` range, and maps them to macroblock-level QP offsets saved as flat, row-major binary files.

### Function-Level Description

#### `mb_grid(native_width=1920, native_height=1080) -> (int, int)`
- **Purpose**: Calculates the size of the macroblock grid (16x16 pixel blocks) for a given resolution.
- **Details**:
  - `num_mb_width = (native_width + 15) // 16`
  - `num_mb_height = (native_height + 15) // 16`

#### `call_model(frame: np.ndarray) -> torch.Tensor`
- **Purpose**: Runs visual saliency model inference on a raw BGR video frame.
- **Details**:
  - Receives a 3D BGR numpy array `[H, W, C]` representing the decoded video frame.
  - Saliency model runs inference (default placeholder generates a horizontal gradient from `0.0` on the left to `1.0` on the right).
  - Returns a 3D `torch.Tensor` of shape `[1, mb_height, mb_width]` containing the saliency values, preserving the batch dimension.

#### `normalize_heatmap(hmap: torch.Tensor, num_mb_width: int, num_mb_height: int) -> np.ndarray`
- **Purpose**: Interpolates and min/max normalizes the saliency map.
- **Details**:
  - Resizes the heatmap to exactly match the macroblock grid using bilinear interpolation (`torch.nn.functional.interpolate`) on the 4D unsqueezed tensor `[B, 1, H, W]`.
  - Calculates the minimum and maximum values.
  - Subtracts the minimum and divides by the range: `(hmap - hmap_min) / (hmap_max - hmap_min)` to scale the saliency values exactly to `[0.0, 1.0]` (preserving full dynamic range for QP mapping).
  - Returns the squeezed 2D numpy array `[mb_height, mb_width]`.

#### `apply_spatial_smoothing(hmap: np.ndarray, sigma: float) -> np.ndarray`
- **Purpose**: Applies Gaussian filtering to smooth out spatial discontinuities.
- **Details**:
  - Uses `scipy.ndimage.gaussian_filter` with the specified `sigma`. Returns the original array if `sigma <= 0.0`.

#### `map_to_qp_offsets(normalized: np.ndarray, qp_min: float, qp_max: float) -> np.ndarray`
- **Purpose**: Linearly maps normalized `[0, 1]` saliency intensities to a range of QP offsets.
- **Details**:
  - Linear mapping formula: `qp_offset = qp_max - (qp_max - qp_min) * normalized`
    - High-saliency macroblocks (saliency → 1.0) receive the **minimum** QP offset (e.g., `-6.0`), conserving details.
    - Low-saliency background macroblocks (saliency → 0.0) receive the **maximum** QP offset (e.g., `+6.0`), compressing them more aggressively.
  - Clips values to `[qp_min, qp_max]` and casts the output to float32.

#### `main(sys_args=None)`
- **Purpose**: Orchestrates the generator process.
- **Details**:
  - Parses CLI inputs (`--video_path`, `--output_dir`, `--qp_min`, `--qp_max`, `--spatial_sigma`, `--temporal_alpha`).
  - Decodes the video using OpenCV `cv2.VideoCapture` and calls `call_model` on each frame.
  - Applies smoothing, normalizes the maps, and maps them to float32 QP offsets.
  - Flattens each map into **row-major (C) order** (raster scan) and writes them as binary `.bin` files (`qpoffset_000000.bin`).
  - Saves `metadata.json` with grid dimensions, frame counts, and mapped bounds.

---

## 2. C Video Encoder: `encoder.c`

This C application reads an input video, decodes it frame-by-frame, associates each frame with a macroblock QP offset map loaded from a binary file, encodes the frames via the x264 encoder, and copies audio streams to compile an output MP4.

### Structures & Helper Functions

#### `EncoderConfig` (struct)
- **Fields**:
  - `input_video`: Path to input video file.
  - `output_mp4`: Path to output MP4 file.
  - `qpoffset_dir`: Directory containing the `.bin` files.
  - `crf`: Constant Rate Factor value (default: `23.0`).
  - `preset`: Speed-preset string for x264 (default: `"medium"`).
  - `tune`: Visual tuning string for x264 (default: `NULL`).

#### `print_usage(const char *program_name)`
- **Purpose**: Outputs syntax/usage help instructions to `stderr`.

#### `parse_arguments(int argc, char **argv, EncoderConfig *cfg)`
- **Purpose**: Parses input parameters and maps CLI arguments into the `EncoderConfig` structure, setting defaults.

### Main Driver Structure (`main`)

The `main` function executes the following pipeline:

#### Step 1: Input Demuxing (FFmpeg libavformat)
1. Calls `avformat_open_input` to open the input video container.
2. Calls `avformat_find_stream_info` to parse file metadata.
3. Iterates over streams to identify the first **Video Stream** and **Audio Stream**.

#### Step 2: Video Decoding Setup (FFmpeg libavcodec)
1. Finds the correct decoder using `avcodec_find_decoder`.
2. Allocates context via `avcodec_alloc_context3`.
3. Copies decoder parameter settings from input stream using `avcodec_parameters_to_context`.
4. Opens the video decoder via `avcodec_open2`.
5. Guesses the video frame rate using `av_guess_frame_rate`.

#### Step 3: x264 Encoder Configuration (x264 API)
1. Initializes `x264_param_t` structure with default parameters based on chosen preset and tune via `x264_param_default_preset`.
2. Overwrites settings to match input resolution, frame rate, and YUV420P format.
3. Configures rate control method: `X264_RC_CRF` with custom baseline CRF value.
4. **Disables mb-tree**: `rc.b_mb_tree = 0`. This is critical. If `b_mb_tree` is active, x264's internal lookahead temporal rate-control override will cancel out any custom macroblock QP offsets set in `pic.prop.quant_offsets`.
5. Sets timebase: `i_timebase_num = fps.den` and `i_timebase_den = fps.num`.
6. Opens the x264 encoder: `x264_encoder_open(&x264_param)`.

#### Step 4: Output Muxer Setup (FFmpeg libavformat)
1. Allocates output MP4 context: `avformat_alloc_output_context2`.
2. **Video Stream Muxing**: Creates an output video stream (`avformat_new_stream`) and copies codec properties.
3. **SPS/PPS Headers Mapping**: Calls `x264_encoder_headers` to extract the binary headers (SPS, PPS, SEI), concatenates them, and assigns them to the video stream's `codecpar->extradata`. This ensures the output file can be parsed correctly by media players.
4. **Audio Stream Muxing**: If the input video has an audio track, creates an output audio stream and copies parameters directly from the input stream (`avcodec_parameters_copy`).
5. Opens the output file (`avio_open`) and writes the file container header (`avformat_write_header`).

#### Step 5: Software Scaler (FFmpeg libswscale)
1. Instantiates `SwsContext` via `sws_getContext`.
2. Converts the decoded frame format (such as YUV422P, YUV444P, or RGB) to YUV420P (`AV_PIX_FMT_YUV420P`) directly into x264 input image planes.

#### Step 6: Core Processing Loop
Reads input packets chronologically using `av_read_frame`:
- **For Audio Packets**: Rescales packet timestamps using `av_packet_rescale_ts` and writes them directly to the output stream via `av_interleaved_write_frame` (zero-overhead stream copy).
- **For Video Packets**:
  1. Sends packet to the decoder via `av_codec_send_packet`.
  2. Receives decoded frames in a loop via `av_codec_receive_frame`.
  3. For each decoded frame:
     - Initializes `x264_picture_t pic_in` and allocates its raw planes using `x264_picture_alloc`.
     - Calls `sws_scale` to fill `pic_in` planes with YUV420P data.
     - Computes the macroblock dimensions for the frame:
       - `mb_width = (width + 15) / 16`
       - `mb_height = (height + 15) / 16`
       - `num_mbs = mb_width * mb_height`
     - Allocates a heap array of size `num_mbs * sizeof(float)` for QP offsets.
     - Opens the binary QP offset file: `<dir>/qpoffset_<frame_count:06d>.bin`.
     - Reads the float32 array into the allocated memory. If file reading fails, defaults the offsets to `0.0`.
     - Passes the pointer to `pic_in.prop.quant_offsets`.
     - Set `pic_in.prop.quant_offsets_free = free` to instruct the x264 encoder to automatically free the allocated heap memory when it is done encoding this frame.
     - Encodes the frame using `x264_encoder_encode`.
     - If the encoder produces output NAL units (frame size > 0):
       - Allocates an `AVPacket` using `av_packet_alloc`.
       - Allocates buffer space via `av_new_packet and copies the NAL payload into it.
       - Maps timestamps from x264 timebase to output stream timebase.
       - Writes packet via `av_interleaved_write_frame`.
     - Frees the allocated structures using `x264_picture_clean`.

#### Step 7: Flush & Clean-up
1. Loops through `x264_encoder_delayed_frames` to retrieve buffered B-frames and write them to output.
2. Writes the file container footer via `av_write_trailer`.
3. Frees all contexts, decoders, scalers, buffers, and exits.

---

## 3. Python QP Visualization: `visualize_qps.py`

This script overlays the macroblock grid and a color-coded QP offset heatmap directly onto the video frames, printing the exact numerical QP offset value inside each cell.

### Key Features
1. **Video Decoding & Upscaling**: Decodes the video frame-by-frame and upscales each frame by a factor of 4 (e.g. 320x240 becomes 1280x960) to ensure the macroblock cells (originally 16x16, now 64x64 pixels) are large enough to fit readable text overlays.
2. **Color-Coded Heatmap Mask**:
   - **Green Overlay**: Represents negative QP offsets (saliency area, quality enhancement).
   - **Red Overlay**: Represents positive QP offsets (background area, aggressive compression).
   - The color intensity is linearly proportional to the offset magnitude.
3. **Numeric Labels**: Renders the exact QP offset value (e.g. `+5.3` or `-12.5`) at the center of each macroblock. Renders text with a black outline and white inner fill for high contrast and readability on any video background.
4. **Header HUD**: Overlays a transparent HUD bar at the top showing the frame count, frame index, and a color legend.
5. **Video Compilation**: Encodes and saves the annotated frames into `sample_visualized.mp4` using OpenCV `VideoWriter`.

---

## 4. Build & Verification System

### CMake Configuration: `CMakeLists.txt`
This file compiles the C encoder and links the necessary dependencies.
1. Checks if the compiler is MinGW/GCC on Windows. If true:
   - Configures search path directives targeting MSYS2 folders: `C:/msys64/mingw64/include` and `C:/msys64/mingw64/lib`.
2. Creates the executable target: `add_executable(encoder encoder.c)`.
3. Links: `avformat`, `avcodec`, `avutil`, `swscale`, and `x264`.
4. If compiled on non-Windows/Unix configurations, uses standard `PkgConfig` modules to find the libraries automatically.

### Python Compilation Script: `build.py`
A small helper to compile `encoder.c` directly without relying on CMake or PowerShell.
- Uses `subprocess.run` to call the MSYS2 Mingw64 compiler `C:\msys64\mingw64\bin\gcc.exe` directly on the host system, passing all include and linking flags.
- Runs without a shell wrapper to avoid environment path redirection issues.

### Verification Script: `generate_test_data.py`
Orchestrates the entire pipeline end-to-end for validation:
1. Calls FFmpeg `testsrc` filter to create a 4-second dummy video (`input_video.mp4`, 320x240, 25fps = 100 frames).
2. Runs `qp_map_generator.py` on `input_video.mp4` to produce binary `.bin` QP offsets in `test_qpoffsets/` (using the model placeholder's gradient).
3. Executes the compiled `encoder.exe` to produce `output_video.mp4`.
4. Runs `visualize_qps.py` to produce `visualized_output.mp4` showing the macroblock QP values.
5. Checks if outputs exist and are non-empty to confirm success.

### Sample Production Pipeline Runner: `run_sample_pipeline.py`
Orchestrates the compilation, preprocessing, rate-controlled encoding, and visual inspection of a production video (`sample_video.mp4`):
1. **Compilation**: Invokes GCC to build `encoder.exe` with runtime dynamic libraries.
2. **Metadata Analysis**: Automatically reads properties of `sample_video.mp4` (dimensions, frame rate, total frames) to calculate macroblock constraints.
3. **Saliency Mapping**: Runs `qp_map_generator.py` to decode frames and map in-memory model saliencies directly to raster-scan float32 binaries in `sample_qpoffsets/`.
4. **Encoding**: Runs `encoder.exe` with a base quality of `--crf 20` using the generated QP boundaries to output `sample_output.mp4`.
5. **Visualization**: Invokes `visualize_qps.py` to render color-coded masks (Green/Red overlays) and text labels of macroblock offsets over the encoded video, saving it as `sample_visualized.mp4`.
6. **Validation**: Verifies filesizes and outputs a status report.